In [1]:
#!/usr/bin/env python3
import argparse
import time
from pathlib import Path

import numpy as np
import serial
from scipy.io import wavfile

In [7]:
fs, pcg = wavfile.read('mediciones_simultaneas/DAVID171225_3.wav')
pcg = pcg[:,0]

/tmp/ipykernel_3532274/1964281964.py:1: WavFileWarning: Chunk (non-data) not understood, skipping it.
  fs, pcg = wavfile.read('mediciones_simultaneas/DAVID171225_3.wav')


In [ ]:
HEADER = b"\xDE\xAD\xBE\xEF"
FS_IN = 48000
N_IN = 49152
DOWNSAMPLE = 48
FS_OUT = FS_IN // DOWNSAMPLE
N_OUT = N_IN // DOWNSAMPLE

CHUNK_USB_BURST = 1024  # 16 paquetes de 64 bytes

s
def wav_read(path: Path) -> tuple[int, np.ndarray]:
    fs, x = wavfile.read(path)
    return fs, x


def adapt_len(x: np.ndarray, n_target: int = N_IN) -> np.ndarray:
    if len(x) < n_target:
        y = np.zeros(n_target, dtype=np.float32)
        y[: len(x)] = x
        x = y
    elif len(x) > n_target:
        x = x[:n_target]

    return x.astype(np.float32, copy=False)


def read_exact(ser: serial.Serial, nbytes: int, timeout_s: float) -> bytes:
    deadline = time.time() + timeout_s
    data = bytearray()
    while len(data) < nbytes:
        if time.time() > deadline:
            raise TimeoutError(f"Timeout leyendo {nbytes} bytes; solo llegaron {len(data)}")
        chunk = ser.read(nbytes - len(data))
        if chunk:
            data.extend(chunk)
    return bytes(data)


def sync_to_header(ser: serial.Serial, timeout_s: float) -> None:
    deadline = time.time() + timeout_s
    window = bytearray()

    while True:
        if time.time() > deadline:
            raise TimeoutError("Timeout esperando cabecera de respuesta")
        b = ser.read(1)
        if not b:
            continue
        window.extend(b)
        if len(window) > len(HEADER):
            window.pop(0)
        if bytes(window) == HEADER:
            return


def send_frame_in_chunks(ser: serial.Serial, frame: bytes, chunk_size: int = CHUNK_USB_BURST) -> None:
    for i in range(0, len(frame), chunk_size):
        part = frame[i : i + chunk_size]
        nw = ser.write(part)
        if nw != len(part):
            raise IOError(f"Escritura incompleta: {nw}/{len(part)} bytes")
    ser.flush()


def receive_processed_vector(ser: serial.Serial, timeout_s: float) -> np.ndarray:
    # El firmware responde: 4 bytes header + 4096 bytes payload (1024 float32)
    sync_to_header(ser, timeout_s=timeout_s)
    payload = read_exact(ser, N_OUT * 4, timeout_s=timeout_s)
    y = np.frombuffer(payload, dtype="<f4").copy()
    if y.size != N_OUT:
        raise ValueError(f"Tamano de salida inesperado: {y.size}, esperado: {N_OUT}")
    return y


def main():
    parser = argparse.ArgumentParser(description="TX/RX PCG con STM32 CDC FS")
    parser.add_argument("--port", required=True, help="Puerto serial, ejemplo: /dev/ttyACM0")
    parser.add_argument("--baud", type=int, default=115200, help="Baudrate (CDC lo ignora en la practica)")
    parser.add_argument("--input", default="mediciones_simultaneas/DAVID171225_3.wav", help="Ruta WAV entrada")
    parser.add_argument("--output", default="pcg_procesado_1k.wav", help="Ruta WAV salida")
    parser.add_argument("--timeout", type=float, default=10.0, help="Timeout en segundos")
    args = parser.parse_args()

    in_path = Path(args.input)
    out_path = Path(args.output)

    if not in_path.exists():
        raise FileNotFoundError(f"No existe archivo de entrada: {in_path}")

    _, x = wav_read(in_path)
    x = adapt_len(x, n_target=N_IN)

    payload = x.astype("<f4", copy=False).tobytes()
    frame_tx = HEADER + payload

    if len(frame_tx) != 4 + N_IN * 4:
        raise RuntimeError(f"Frame TX invalido: {len(frame_tx)} bytes")

    with serial.Serial(
        port=args.port,
        baudrate=args.baud,
        timeout=0.05,
        write_timeout=2.0,
        exclusive=True,
    ) as ser:
        # Limpia buffers por si hay datos antiguos
        ser.reset_input_buffer()
        ser.reset_output_buffer()

        t0 = time.time()
        send_frame_in_chunks(ser, frame_tx, chunk_size=CHUNK_USB_BURST)
        y = receive_processed_vector(ser, timeout_s=args.timeout)
        t1 = time.time()

    wavfile.write(out_path, FS_OUT, y.astype(np.float32))
    print(f"Listo. Archivo salida: {out_path}")
    print(f"Muestras salida: {len(y)} a {FS_OUT} Hz")
    print(f"Tiempo total TX+PROC+RX: {1000*(t1-t0):.1f} ms")


if __name__ == "__main__":
    main()